# Policy Search Assistant v2 — Setup (copied from v1, unchanged)

In [1]:
import os
from google import genai
from dotenv import load_dotenv, find_dotenv
from pypdf import PdfReader
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
client=genai.Client(api_key=api_key)
reader=PdfReader("CFPB Udaap.pdf")
document_text=""
for page in reader.pages:
    document_text+=page.extract_text()


def chunk_text(text, chunk_size=800, overlap=100):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters shared between consecutive chunks (keeps context from being cut mid-sentence)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward, but overlap with previous chunk
    return chunks

chunks = chunk_text(document_text)
print(f"Number of chunks: {len(chunks)}")
print("--- First chunk ---")
print(chunks[0])
print("--- Second chunk ---")
print(chunks[1])
print(len(chunks))

def embed_chunks(chunks, client, model="gemini-embedding-001"):
    """
    Generates an embedding vector for each text chunk using Gemini's embedding model.
    Returns a list of vectors, same order as input chunks.
    """
    embeddings = []
    for chunk in chunks:
        result = client.models.embed_content(
            model=model,
            contents=chunk
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings

embeddings = embed_chunks(chunks, client)
print(f"Number of embeddings: {len(embeddings)}")
print(f"Vector length (dimensions): {len(embeddings[0])}")
print(f"First 5 values of first vector: {embeddings[0][:5]}")

import faiss
import numpy as np

embeddings_np = np.array(embeddings).astype('float32')

# Normalize vectors to unit length — this is what makes inner product == cosine similarity
faiss.normalize_L2(embeddings_np)

dimension = embeddings_np.shape[1]
index = faiss.IndexFlatIP(dimension)  # IP = inner product

index.add(embeddings_np)

print(f"Number of vectors in index: {index.ntotal}")  # should print 63

chunk_id_to_text = {i: chunks[i] for i in range(len(chunks))}

print(f"Number of entries in mapping: {len(chunk_id_to_text)}")
print(f"Sample entry (id 0): {chunk_id_to_text[0][:150]}...")

def search_policy(query, index, chunk_id_to_text, client, model="gemini-embedding-001", top_k=3):
    """
    Embeds a query, searches the FAISS index, and returns the top_k matching chunks.
    """
    # 1. Embed the query the same way chunks were embedded
    result = client.models.embed_content(model=model, contents=query)
    query_vector = np.array([result.embeddings[0].values]).astype('float32')

    # 2. Normalize the query vector — required for cosine-equivalence with IndexFlatIP
    faiss.normalize_L2(query_vector)

    # 3. Search the index
    scores, indices = index.search(query_vector, top_k)

    # 4. Map FAISS index positions back to chunk text
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "chunk_id": int(idx),
            "score": float(score),
            "text": chunk_id_to_text[idx] if isinstance(list(chunk_id_to_text.keys())[0], int) else chunk_id_to_text[str(idx)]
        })
    return results

# Test it
test_query = "What counts as an unfair act under UDAAP?"
results = search_policy(test_query, index, chunk_id_to_text, client)

for r in results:
    print(f"\n--- Chunk {r['chunk_id']} (score: {r['score']:.4f}) ---")
    print(r['text'][:300])



Number of chunks: 63
--- First chunk ---
CFPB Consumer 
Laws and Regulations UDAAP 
CFPB Manual V.2 (October 2012) UDAAP 1 
Unfair, Deceptive, or Abusive Acts or Practices 
Unfair, deceptive, or abusive acts and practices (UDAAPs) can cause significant financial injury to 
consumers, erode consumer confidence, and undermine the financial marketplace. Under the 
Dodd-Frank Act, it is unlawful for any provider of consumer financial products or services or a 
service provider to engage in any unfair, deceptive or abusive act or practice.1 The Act also 
provides CFPB with rule-making authority and, with respect to entities within its jurisdiction, 
enforcement authority to prevent unfair, deceptive, or abusive acts or practices in connection with 
any transaction with a consumer for a consumer financial product or service, or the off
--- Second chunk ---
ection with 
any transaction with a consumer for a consumer financial product or service, or the offering of a 
consumer financial produc

In [ ]:
# =============================================================
# 7A.1 BASELINE EVAL: QUERIES + RETRIEVAL INSPECTION
# What this does:
#   1. Defines the 6 test queries (direct, vague, mismatch, out-of-scope).
#   2. Defines retrieve(), a wrapper around search_policy() so later steps
#      (multi-query, HyDE) can swap in their own retrieval without touching eval code.
#   3. Prints the top-5 chunks for each query with scores, so you can
#      label by hand which chunks actually answer each question.
#   4. Defines find_chunks(), a keyword search across all chunks, for
#      finding the right chunk when it's not in the top 5.
# Output: reference material for labeling. It doesn't compute any metrics.
# =============================================================


eval_queries = [
    {"id": "q1", "type": "direct",       "query": "What counts as an unfair act under UDAAP?"},
    {"id": "q2", "type": "direct",       "query": "How does UDAAP define an abusive practice?"},
    {"id": "q3", "type": "vague",        "query": "Can we hide fees in the fine print?"},
    {"id": "q4", "type": "vague",        "query": "Is it okay to pressure customers into signing up quickly?"},
    {"id": "q5", "type": "mismatch",     "query": "What makes a disclosure misleading to a reasonable consumer?"},
    {"id": "q6", "type": "out_of_scope", "query": "What is the minimum credit score for a mortgage?"},
]

K = 5

def retrieve(query, k=K):
    return search_policy(query, index, chunk_id_to_text, client, top_k=k)

for q in eval_queries:
    print(f"\n=== {q['id']} [{q['type']}] {q['query']}")
    for r in retrieve(q["query"]):
        snippet = r["text"][:250].replace("\n", " ")
        print(f"  chunk {r['chunk_id']:>2} | {r['score']:.3f} | {snippet}...")

def find_chunks(keyword):
    return [i for i, t in chunk_id_to_text.items() if keyword.lower() in t.lower()]
print(find_chunks("abusive"))


=== q1 [direct] What counts as an unfair act under UDAAP?
  chunk  0 | 0.781 | CFPB Consumer  Laws and Regulations UDAAP  CFPB Manual V.2 (October 2012) UDAAP 1  Unfair, Deceptive, or Abusive Acts or Practices  Unfair, deceptive, or abusive acts and practices (UDAAPs) can cause significant financial injury to  consumers, erode ...
  chunk  3 | 0.759 | Act is that an act or practice is unfair when:  (1) It causes or is likely to cause substantial injury to consumers; (2) The injury is not reasonably avoidable by consumers; and 1 Dodd-Frank Act, Title X, Subtitle C, Sec. 1036; PL 111-203 (July 21, 2...
  chunk  8 | 0.741 |                                                                    4 The standard for unfairness in the Dodd-Frank Act has the same three-part test as the FTC Act. This standard was first stated in  the FTC Policy Statement on Unfairness (Dec. 17, 19...
  chunk 50 | 0.739 | ing or other changes in terms.  d. A product combines features and terms in a manner that can i

In [3]:
# =============================================================
# 7A.1 LABELING HELPER: READ CANDIDATE CHUNKS + KEYWORD CHECKS
# What this does:
#   1. Prints the full text of the candidate chunks (picked from the
#      baseline top-5 output), so you can decide by reading which ones
#      actually contain the answer to each eval query.
#   2. Keyword-searches the whole document for terms related to q4
#      ("pressure", "urgent", etc.) to check whether the document covers
#      that topic directly at all.
# Output: reading material for labeling. It doesn't compute any metrics
# and nothing from it is saved.
# =============================================================

# Chunk IDs to read in full (from the baseline top-5s plus their neighbors)
candidates = [3, 4, 5, 33, 34, 35, 18, 19, 21, 22, 27, 28, 53]
for cid in candidates:
    print(f"\n===== chunk {cid} =====")
    print(chunk_id_to_text[cid])  # full chunk text, not just a snippet

# Does the document mention these q4-related terms anywhere? (empty list = no)
for kw in ["pressure", "unreasonable advantage", "take advantage", "urgent"]:
    print(kw, "->", find_chunks(kw))  # find_chunks() is defined in the earlier cell


===== chunk 3 =====
Act is that an act or practice is unfair when: 
(1) It causes or is likely to cause substantial injury to consumers;
(2) The injury is not reasonably avoidable by consumers; and
1 Dodd-Frank Act, Title X, Subtitle C, Sec. 1036; PL 111-203 (July 21, 2010).
2 Sec. 1031 of the Dodd-Frank Act. The principles of “unfair” and “deceptive” practices in the Act are similar to those under Sec.
5 of the Federal Trade Commission Act (FTC Act). The Federal Trade Commission (FTC) and federal banking regulators have 
applied these standards through case law, official policy statements, guidance, examination procedures, and enforcement actions 
that may inform CFPB. 
3 Dodd-Frank Act, Secs. 1024; 1025(b)(1); 1026(b) of the Act.CFPB Consumer  
Laws and Regulations  UDAAP 
CFPB Manual V.2 (October 2012) UDA

===== chunk 4 =====
; 1026(b) of the Act.CFPB Consumer  
Laws and Regulations  UDAAP 
CFPB Manual V.2 (October 2012) UDAAP 2 
(3) The injury is not outweighed by countervailing 

In [5]:
#python
# =============================================================
# 7A.1 BASELINE EVALUATION + SAVE LABELS
# What this does:
#   1. Stores your hand-labeled relevant chunk IDs per query (the "answer key").
#   2. Defines evaluate(), which runs any retrieval function over all eval
#      queries and reports Hit@5 (did a correct chunk appear?) and
#      MRR (how high did the first correct chunk rank?).
#   3. Runs it on the plain FAISS retrieval to get the baseline numbers.
#   4. Saves queries, labels and a text anchor per chunk to eval_set.json,
#      so labels can be re-mapped if chunk IDs change later.
# =============================================================

import json

relevant = {
    "q1": [3, 4],      # three-part unfairness test (split across chunks)
    "q2": [33, 34],    # abusive definition (continues into 34)
    "q3": [19, 27, 53],# fine print can't cure misleading claims / FTC example / disclose fees prominently
    "q4": [33],        # closest match: abusive = unreasonable advantage (inferred, no direct coverage)
    "q5": [18, 22],    # deception test / reasonable consumer perspective
    "q6": [],          # out of scope
}

def evaluate(retrieve_fn, label, k=K):
    print(f"\n--- {label} ---")
    hits, rrs = [], []
    for q in eval_queries:
        results = retrieve_fn(q["query"], k=k)
        ids = [r["chunk_id"] for r in results]
        top = results[0]["score"]
        rel = relevant[q["id"]]
        if rel:
            ranks = [ids.index(c) + 1 for c in rel if c in ids]
            hit = bool(ranks)
            rr = 1 / min(ranks) if ranks else 0.0
            hits.append(hit); rrs.append(rr)
            print(f"{q['id']} {q['type']:<13} hit@{k}={hit!s:<5} rr={rr:.2f} top_score={top:.3f}")
        else:
            print(f"{q['id']} {q['type']:<13} out-of-scope      top_score={top:.3f}")
    print(f"Hit@{k}: {sum(hits)}/{len(hits)} | MRR: {sum(rrs)/len(rrs):.3f}")

evaluate(retrieve, "baseline (FAISS, plain query)")

# Save labels with a text anchor per chunk, so they can be re-mapped if chunk IDs change later
anchors = {cid: chunk_id_to_text[cid][350:410].replace("\n", " ")
           for ids in relevant.values() for cid in ids}
with open("eval_set.json", "w", encoding="utf-8") as f:
    json.dump({"queries": eval_queries, "relevant": relevant, "anchors": anchors}, f, indent=2)


--- baseline (FAISS, plain query) ---
q1 direct        hit@5=True  rr=0.50 top_score=0.781
q2 direct        hit@5=True  rr=0.50 top_score=0.779
q3 vague         hit@5=True  rr=1.00 top_score=0.668
q4 vague         hit@5=False rr=0.00 top_score=0.602
q5 mismatch      hit@5=True  rr=1.00 top_score=0.782
q6 out_of_scope  out-of-scope      top_score=0.541
Hit@5: 4/5 | MRR: 0.600


In [8]:
# =============================================================
# 7A.2 MULTI-QUERY GENERATION (fixed: thinking off + retry)
# What this does: asks Gemini to rewrite one user question into
# several differently-worded versions, using regulatory vocabulary
# where that fits. Only generates the variants; it doesn't search
# yet (that's 7A.3).
# Changes from the first version:
#   - thinking_budget=0: stops hidden reasoning tokens from eating
#     the output limit (this was truncating the variants)
#   - retry with a short wait on ServerError (transient API overloads)
# # Change 2: output cleaning (regex strips numbering/bullets, drops heading lines)
#   because Gemini sometimes adds "1." or a header despite being told not to
# =============================================================
import time
import re
from google.genai import types, errors

def generate_query_variants(query, n=3, model="gemini-2.5-flash", retries=3):
    prompt = f"""You are helping search a CFPB UDAAP (Unfair, Deceptive, or Abusive
Acts or Practices) examination manual.

Rewrite the user question below into {n} alternative search queries.
Rules:
- Each version should be worded differently from the others and from the original.
- Use formal regulatory vocabulary where it fits (e.g. "material information",
  "reasonable consumer", "abusive", "unfair", "deceptive", "disclosure").
- Keep the meaning of the original question.
- Return ONLY the {n} queries, one per line, with no numbering or extra text.

User question: {query}"""
    config = types.GenerateContentConfig(
        temperature=0.7,
        max_output_tokens=1024,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
            break
        except errors.ServerError as e:
            if attempt == retries - 1:
                raise
            wait = 2 ** (attempt + 1)  # 2s, then 4s
            print(f"  (server error, retrying in {wait}s...)")
            time.sleep(wait)
    lines = []
    for l in response.text.strip().split("\n"):
        l = re.sub(r'^\s*(\d+[.)]|[-•*])\s*', '', l).strip(' "')  # removes "1.", "2)", "-", "•"
        if l and not l.startswith("<") and not l.endswith(":"):    # drops "<h2>...</h2>" or "Queries:" lines
            lines.append(l)
    return lines[:n]

# Test it on all eval queries
for q in eval_queries:
    print(f"\n{q['id']} original: {q['query']}")
    for i, v in enumerate(generate_query_variants(q["query"]), 1):
        print(f"   v{i}: {v}")


q1 original: What counts as an unfair act under UDAAP?
   v1: What are the criteria for an act or practice to be deemed unfair under UDAAP?
   v2: Under what circumstances does the CFPB determine a practice to be an unfair act within the UDAAP framework?
   v3: What constitutes an unfair act or practice as defined by the UDAAP provisions?

q2 original: How does UDAAP define an abusive practice?
   v1: What are the characteristics of an abusive act or practice under UDAAP?
   v2: What constitutes an abusive practice as defined by UDAAP regulations?
   v3: Can you elaborate on the criteria for identifying an abusive practice within the scope of UDAAP?

q3 original: Can we hide fees in the fine print?
   v1: Is it permissible to obscure additional charges within the terms and conditions?
   v2: Would concealing material fee information in dense disclosures constitute an unfair or deceptive practice?
   v3: Does failing to prominently disclose all applicable fees in a clear manner violate